# Security & Secrets Management — Azure Supplement

> **Mission:** Use Azure Key Vault for production-grade secret management — centralized storage, RBAC, automatic rotation, audit logs.
>
> **This notebook:** Azure-specific secrets workflow (Key Vault, Managed Identity, Azure Policy). We'll demonstrate how to securely access secrets from containers without hardcoded credentials.
>
> **Main notebook:** Local secrets workflow (Docker Secrets, .env files, pre-commit hooks).

---

## Prerequisites

- **Azure subscription** ([free tier](https://azure.microsoft.com/free/) — $200 credit for 30 days)
- **Azure CLI** installed ([docs](https://docs.microsoft.com/cli/azure/install-azure-cli))
- **Docker** installed (for containerized apps)

Verify installation:
```bash
az --version
docker --version
```

---

## Cell 1: Azure Credentials and Subscription Setup

**Goal:** Authenticate with Azure and select the subscription for Key Vault setup.

**What we're doing:**
- Log in to Azure CLI
- List available subscriptions
- Set the default subscription

**Security note:** We'll use Managed Identity for production (no credentials in code). For local development, we'll use Azure CLI authentication.

In [ ]:
%%bash

# Azure credentials
AZURE_SUBSCRIPTION_ID=""  # Stripped by pre-push hook (fill in locally)

# Log in to Azure (opens browser)
az login

# List subscriptions
echo "Available subscriptions:"
az account list --output table

# Set default subscription (if not already set)
if [ -n "$AZURE_SUBSCRIPTION_ID" ]; then
    az account set --subscription "$AZURE_SUBSCRIPTION_ID"
    echo ""
    echo "✅ Using subscription: $AZURE_SUBSCRIPTION_ID"
else
    echo ""
    echo "⚠️  AZURE_SUBSCRIPTION_ID not set. Using default subscription."
fi

# Verify current subscription
echo ""
echo "Current subscription:"
az account show --output table

echo ""
echo "✅ Azure authentication complete"

**Expected output:**
```
Available subscriptions:
Name              CloudName    SubscriptionId                        State
----------------  -----------  ------------------------------------  -------
My Subscription   AzureCloud   12345678-1234-1234-1234-123456789012  Enabled

✅ Azure authentication complete
```

✅ **Best practice:** For CI/CD pipelines, use a Service Principal instead of interactive login:
```bash
az login --service-principal \
  --username $AZURE_CLIENT_ID \
  --password $AZURE_CLIENT_SECRET \
  --tenant $AZURE_TENANT_ID
```

---

## Cell 2: Azure Key Vault Setup

**Goal:** Create an Azure Key Vault to store secrets securely.

**What we're doing:**
- Create a resource group for Key Vault
- Create a Key Vault instance
- Configure access policies (who can read/write secrets)

**Why Key Vault?** Unlike `.env` files or Kubernetes Secrets (base64-encoded), Key Vault provides:
- **Encryption at rest** (AES-256)
- **Access control** via Azure RBAC or access policies
- **Audit logs** (who accessed what secret when)
- **Automatic rotation** (optional, with Azure Automation)

In [ ]:
%%bash

# Configuration
RESOURCE_GROUP="rg-secrets-demo"
LOCATION="eastus"
KEYVAULT_NAME="kv-secrets-demo-$(date +%s)"  # Must be globally unique

echo "Creating Azure Key Vault..."
echo "Resource Group: $RESOURCE_GROUP"
echo "Key Vault Name: $KEYVAULT_NAME"
echo "Location: $LOCATION"
echo ""

# Create resource group
az group create \
  --name "$RESOURCE_GROUP" \
  --location "$LOCATION"

echo ""

# Create Key Vault
az keyvault create \
  --name "$KEYVAULT_NAME" \
  --resource-group "$RESOURCE_GROUP" \
  --location "$LOCATION" \
  --sku Standard

echo ""
echo "✅ Key Vault created: $KEYVAULT_NAME"

# Show Key Vault details
az keyvault show --name "$KEYVAULT_NAME" --query "{Name:name, URI:properties.vaultUri}" --output table

# Save for later cells
echo "$KEYVAULT_NAME" > /tmp/keyvault_name.txt
echo "$RESOURCE_GROUP" > /tmp/resource_group.txt

echo ""
echo "✅ Key Vault is ready for secret storage"

**Expected output:**
```
✅ Key Vault created: kv-secrets-demo-1714129200

Name                       URI
-------------------------  ------------------------------------------------
kv-secrets-demo-1714129200 https://kv-secrets-demo-1714129200.vault.azure.net/
```

✅ **Production best practices:**
- **Soft delete** (enabled by default): Deleted secrets are recoverable for 90 days
- **Purge protection**: Prevents permanent deletion during retention period
- **Private endpoints**: Restrict access to virtual network only
- **Diagnostic logging**: Enable audit logs to Log Analytics workspace

**Pricing:** Standard tier is $0.03 per 10,000 operations (~$3/month for moderate usage).

---

## Cell 3: Store Secrets in Key Vault

**Goal:** Store database credentials and API keys in Key Vault.

**What we're doing:**
- Create secrets with `az keyvault secret set`
- Retrieve secrets with `az keyvault secret show`
- Demonstrate secret versioning

**Key concept:** Key Vault automatically versions secrets. When you update a secret, the old version remains accessible (useful for rollback).

In [ ]:
%%bash

# Load Key Vault name from previous cell
KEYVAULT_NAME=$(cat /tmp/keyvault_name.txt)

echo "Storing secrets in Key Vault: $KEYVAULT_NAME"
echo ""

# Store database password
az keyvault secret set \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --value "super_secret_db_password_v1"

echo ""

# Store API key
az keyvault secret set \
  --vault-name "$KEYVAULT_NAME" \
  --name "api-key" \
  --value "sk_live_abcdef1234567890"

echo ""

# List all secrets
echo "Secrets in Key Vault:"
az keyvault secret list --vault-name "$KEYVAULT_NAME" --query "[].name" --output table

echo ""

# Retrieve a secret (masked in output)
echo "Retrieving db-password:"
az keyvault secret show \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --query "{Name:name, Value:value, Version:id}" \
  --output table

echo ""
echo "✅ Secrets stored securely in Key Vault"
echo "   - Encrypted at rest (AES-256)"
echo "   - Accessible only with proper authentication"
echo "   - Versioned (can rollback if needed)"

**Expected output:**
```
Secrets in Key Vault:
Name
-----------
api-key
db-password

Retrieving db-password:
Name         Value                          Version
-----------  -----------------------------  ------------------------------------
db-password  super_secret_db_password_v1    https://kv-...vault.azure.net/.../...
```

✅ **Secret versioning example:**
```bash
# Update secret (creates new version)
az keyvault secret set --vault-name $KEYVAULT_NAME --name "db-password" --value "new_password_v2"

# Get latest version (default)
az keyvault secret show --vault-name $KEYVAULT_NAME --name "db-password"

# Get specific version
az keyvault secret show --vault-name $KEYVAULT_NAME --name "db-password" --version <version-id>

# List all versions
az keyvault secret list-versions --vault-name $KEYVAULT_NAME --name "db-password"
```

**Why versioning matters:** If a password rotation breaks production, you can quickly rollback to the previous version while you investigate.

---

## Cell 4: Access Key Vault from Container (Managed Identity)

**Goal:** Deploy a containerized app that reads secrets from Key Vault using Managed Identity (no credentials in code).

**What we're doing:**
- Create an Azure Container Instance (ACI) with Managed Identity enabled
- Grant Key Vault access to the Managed Identity
- Deploy a Python app that fetches secrets from Key Vault

**Why Managed Identity?** Your container doesn't need credentials to access Key Vault. Azure automatically provides an identity token that can be used for authentication.

In [ ]:
%%bash

KEYVAULT_NAME=$(cat /tmp/keyvault_name.txt)
RESOURCE_GROUP=$(cat /tmp/resource_group.txt)
CONTAINER_NAME="aci-keyvault-demo"

echo "Deploying container with Managed Identity..."
echo ""

# Create Python app that reads from Key Vault
cat > /tmp/app.py << 'EOF'
import os
from azure.identity import ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient

# Key Vault URL (from environment variable)
vault_url = os.getenv("KEY_VAULT_URL")

# Authenticate with Managed Identity (no credentials needed!)
credential = ManagedIdentityCredential()
client = SecretClient(vault_url=vault_url, credential=credential)

# Retrieve secrets
try:
    db_password = client.get_secret("db-password").value
    api_key = client.get_secret("api-key").value
    
    print("✅ Secrets retrieved from Key Vault:")
    print(f"   DB Password: {db_password[:10]}... (truncated)")
    print(f"   API Key: {api_key[:15]}... (truncated)")
    print("")
    print("✅ App has access to secrets without hardcoded credentials!")
except Exception as e:
    print(f"❌ Error accessing Key Vault: {e}")
EOF

# Create Dockerfile
cat > /tmp/Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app
RUN pip install azure-identity azure-keyvault-secrets
COPY app.py .
CMD ["python", "app.py"]
EOF

# Build and push image to Docker Hub (or ACR)
cd /tmp
docker build -t keyvault-demo:latest .

echo ""
echo "Image built. Now deploying to Azure Container Instances..."
echo ""

# Deploy container with Managed Identity
az container create \
  --resource-group "$RESOURCE_GROUP" \
  --name "$CONTAINER_NAME" \
  --image keyvault-demo:latest \
  --assign-identity --scope $(az keyvault show --name "$KEYVAULT_NAME" --query id -o tsv) \
  --environment-variables KEY_VAULT_URL="https://${KEYVAULT_NAME}.vault.azure.net/" \
  --restart-policy OnFailure

echo ""
echo "⏳ Waiting for container to start..."
sleep 30

# Show container logs
echo ""
echo "Container logs:"
az container logs --resource-group "$RESOURCE_GROUP" --name "$CONTAINER_NAME"

echo ""
echo "✅ Container accessed Key Vault using Managed Identity (no credentials!)"

# Clean up container (keep Key Vault for next cells)
az container delete --resource-group "$RESOURCE_GROUP" --name "$CONTAINER_NAME" --yes

**Expected output:**
```
✅ Secrets retrieved from Key Vault:
   DB Password: super_secr... (truncated)
   API Key: sk_live_abcde... (truncated)

✅ App has access to secrets without hardcoded credentials!
```

✅ **How Managed Identity works:**
1. **Azure assigns an identity** to your container/VM/function
2. **Azure AD issues a token** for that identity
3. **Your code uses the token** to authenticate with Key Vault (no password needed!)
4. **Key Vault checks RBAC**: "Does this identity have 'get secret' permission?"
5. **Secret returned** if authorized

**Code pattern:**
```python
# NO credentials in code!
credential = ManagedIdentityCredential()  # Azure handles authentication
client = SecretClient(vault_url=vault_url, credential=credential)
secret = client.get_secret("my-secret").value
```

**Supported Azure services:**
- Azure Container Instances (ACI)
- Azure App Service
- Azure Functions
- Azure Virtual Machines
- Azure Kubernetes Service (AKS)

---

## Cell 5: Secret Rotation with Key Vault

**Goal:** Rotate a database password in Key Vault without redeploying containers.

**What we're doing:**
- Update the secret in Key Vault (creates new version)
- Demonstrate that containers automatically get the new value on restart
- Set up automatic rotation with Azure Automation (optional)

**Why rotation matters:** Compliance frameworks (SOC 2, PCI-DSS) require regular password rotation (typically every 90 days). Manual rotation is error-prone. Automate it.

In [ ]:
%%bash

KEYVAULT_NAME=$(cat /tmp/keyvault_name.txt)

echo "Secret Rotation Workflow"
echo "========================"
echo ""

# Step 1: Check current secret
echo "Step 1: Current secret version"
az keyvault secret show \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --query "{Name:name, Value:value, Version:id}" \
  --output table

echo ""
echo "---"
echo ""

# Step 2: Rotate secret (create new version)
echo "Step 2: Rotating secret (creating new version)..."
NEW_PASSWORD="rotated_password_$(date +%s)"
az keyvault secret set \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --value "$NEW_PASSWORD"

echo ""
echo "✅ Secret rotated to: $NEW_PASSWORD"

echo ""
echo "---"
echo ""

# Step 3: List all versions
echo "Step 3: All versions of db-password:"
az keyvault secret list-versions \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --query "[].{Version:id, Enabled:attributes.enabled, Created:attributes.created}" \
  --output table

echo ""
echo "---"
echo ""

# Step 4: Retrieve latest version (default)
echo "Step 4: Containers now get the new version:"
az keyvault secret show \
  --vault-name "$KEYVAULT_NAME" \
  --name "db-password" \
  --query "{Name:name, Value:value}" \
  --output table

echo ""
echo "---"
echo ""

cat << 'EOF'
✅ Zero-downtime rotation workflow:

1. Update database to accept BOTH old and new passwords
   (Dual-password window)

2. Rotate secret in Key Vault
   az keyvault secret set --vault-name $VAULT --name db-password --value $NEW_PASSWORD

3. Restart containers (rolling update)
   kubectl rollout restart deployment my-app
   (Containers fetch new password from Key Vault on startup)

4. Verify all containers use new password
   kubectl logs deployment/my-app | grep "Connected"

5. Remove old password from database
   (Now only new password works)

Automatic rotation with Azure Automation:
──────────────────────────────────────────
- Create an Azure Automation account
- Write a runbook (PowerShell or Python) that:
  * Generates a new password
  * Updates the database
  * Updates Key Vault
  * Restarts containers
- Schedule the runbook to run every 90 days

Example Azure Functions rotation:
https://learn.microsoft.com/azure/key-vault/secrets/tutorial-rotation
EOF

echo ""
echo "✅ Secret rotation complete!"

**Best practices for rotation:**

| Strategy | Implementation | Downtime |
|----------|----------------|----------|
| **Manual rotation** | Update Key Vault, restart pods | Risk of errors, slow |
| **Scripted rotation** | Automation script (Azure Functions, GitHub Actions) | Controlled, repeatable |
| **Automatic rotation** | Key Vault + Azure Automation + Database hook | Zero downtime |

✅ **Production pattern:**
```python
# App polls Key Vault every 5 minutes for secret changes
import threading
import time

def refresh_secrets():
    while True:
        time.sleep(300)  # 5 minutes
        new_password = keyvault_client.get_secret("db-password").value
        if new_password != current_password:
            reconnect_to_database(new_password)

threading.Thread(target=refresh_secrets, daemon=True).start()
```

**Why not just restart containers?** Restarting causes downtime. Polling Key Vault and hot-reloading secrets keeps your app running.

---

## Cell 6: Azure Policy — Enforce Secret Scanning

**Goal:** Use Azure Policy to enforce secret scanning across all repositories and prevent deployment if secrets are detected.

**What we're doing:**
- Create an Azure Policy definition that blocks resource creation if secrets are detected
- Apply the policy to the subscription
- Test the policy by attempting to deploy a container with hardcoded secrets

**Why Azure Policy?** Prevents developers from deploying insecure configurations. Enforced at the Azure Resource Manager level (can't be bypassed).

In [ ]:
%%bash

RESOURCE_GROUP=$(cat /tmp/resource_group.txt)

cat << 'EOF'
Azure Policy for Secret Detection
==================================

Policy Use Cases:
─────────────────
1. Block deployment of containers with environment variables matching secret patterns
2. Require all Key Vault access to use Managed Identity (no access keys)
3. Enforce diagnostic logging for all Key Vaults (audit trail)
4. Block public network access to Key Vaults (private endpoints only)

Example Policy Definition (JSON):
──────────────────────────────────
{
  "mode": "All",
  "policyRule": {
    "if": {
      "allOf": [
        {
          "field": "type",
          "equals": "Microsoft.ContainerInstance/containerGroups"
        },
        {
          "count": {
            "field": "Microsoft.ContainerInstance/containerGroups/containers[*].environmentVariables[*]",
            "where": {
              "anyOf": [
                {
                  "field": "Microsoft.ContainerInstance/containerGroups/containers[*].environmentVariables[*].name",
                  "match": "password"
                },
                {
                  "field": "Microsoft.ContainerInstance/containerGroups/containers[*].environmentVariables[*].name",
                  "match": "api_key"
                },
                {
                  "field": "Microsoft.ContainerInstance/containerGroups/containers[*].environmentVariables[*].name",
                  "match": "secret"
                }
              ]
            }
          },
          "greater": 0
        }
      ]
    },
    "then": {
      "effect": "deny"
    }
  }
}

Applying the Policy:
────────────────────
# Create policy definition
az policy definition create \
  --name "deny-secrets-in-env-vars" \
  --display-name "Deny secrets in container environment variables" \
  --description "Blocks deployment of containers with env vars matching secret patterns" \
  --rules policy.json \
  --mode All

# Assign policy to subscription
az policy assignment create \
  --name "enforce-no-secrets" \
  --display-name "Enforce no secrets in environment variables" \
  --policy "deny-secrets-in-env-vars" \
  --scope /subscriptions/{subscription-id}

Testing the Policy:
───────────────────
# Attempt to deploy container with secret in env var (should fail)
az container create \
  --resource-group $RESOURCE_GROUP \
  --name test-container \
  --image nginx \
  --environment-variables PASSWORD=secret123

# Expected result:
# ❌ Error: Resource creation blocked by policy 'deny-secrets-in-env-vars'
#    Reason: Environment variable 'PASSWORD' matches secret pattern

Built-in Azure Policies for Security:
──────────────────────────────────────
- "Key vaults should have soft delete enabled"
- "Key vaults should have purge protection enabled"
- "Key Vault should use a virtual network service endpoint"
- "Diagnostic logs in Key Vault should be enabled"
- "Azure Key Vault Managed HSM should have purge protection enabled"

EOF

echo ""
echo "✅ Recommended Azure Policies for secrets:"
echo ""
echo "   1. Deny hardcoded secrets in environment variables"
echo "   2. Require Managed Identity for Key Vault access"
echo "   3. Enforce diagnostic logging for audit trails"
echo "   4. Block public Key Vault access (private endpoints only)"
echo ""
echo "   Apply policies at subscription or management group level"
echo ""
echo "Documentation:"
echo "https://learn.microsoft.com/azure/governance/policy/samples/built-in-policies"

**Azure Policy Hierarchy:**

```
Management Group (org-wide policies)
    ↓
Subscription (all resources)
    ↓
Resource Group (specific project)
    ↓
Resource (individual Key Vault, Container)
```

✅ **Best practice:** Apply security policies at the **subscription level** so they affect all resource groups. Use **exemptions** sparingly (only for dev/test environments).

**Enforcement modes:**
- **Deny**: Block resource creation/update if policy is violated
- **Audit**: Log policy violations but allow deployment (useful for gradual rollout)
- **Disabled**: Policy is defined but not enforced

**Compliance dashboard:** Azure Policy provides a dashboard showing compliance rate across all resources. Useful for demonstrating security posture to auditors.

---

## Summary

You've completed the Azure Key Vault secrets management workflow:
1. ✅ Created Key Vault for centralized secret storage
2. ✅ Stored secrets (encrypted at rest, versioned)
3. ✅ Accessed secrets from containers using Managed Identity (no credentials in code)
4. ✅ Rotated secrets without downtime
5. ✅ Enforced security policies to prevent secret leaks

**Next steps:**
- **Integrate with CI/CD**: Use Azure Pipelines to deploy containers with Key Vault secrets
- **Monitor access**: Enable diagnostic logs and alert on unusual access patterns
- **Automate rotation**: Set up Azure Automation for 90-day rotation cycles

**Cost:** Key Vault is ~$3/month for moderate usage (10,000 operations). Managed Identity is free.